# DME-Encoder — Kaggle Training Notebook

Полный pipeline: preprocessing → pretraining → fine-tuning → low-label experiment → results.

In [ ]:
# Cell 1 — Setup & Install
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e",
     "/kaggle/working/denoising-event-sequences", "--quiet"],
    check=True,
)
sys.path.insert(0, "/kaggle/working/denoising-event-sequences")

import json
import os
import pathlib
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print(f"Device      : {device}")
print(f"Mixed prec  : {USE_AMP}")
print(f"PyTorch     : {torch.__version__}")

In [ ]:
# Cell 2 — Paths & Config
WORKING_DIR = pathlib.Path("/kaggle/working")
DATA_DIR    = WORKING_DIR / "data"
OUTPUT_DIR  = WORKING_DIR / "outputs"
CKPT_DIR    = OUTPUT_DIR / "checkpoints"
LOG_DIR     = OUTPUT_DIR / "logs"

for p in [DATA_DIR, OUTPUT_DIR, CKPT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

STEP_CKPT_PATH  = CKPT_DIR / "pretrain_step_ckpt.pt"
FINAL_CKPT_PATH = CKPT_DIR / "pretrain_best.pt"
PREP_PATH       = CKPT_DIR / "preprocessor.pkl"
SPLITS_PATH     = CKPT_DIR / "splits.json"

config = {
    "data": {
        "max_seq_len": 256,
        "min_seq_len": 2,
        "event_type_col": "MCC",
        "timestamp_col": "TRDATETIME",
        "target_col": "target_flag",
        "numerical_cols": ["amount"],
        "categorical_cols": ["channel_type", "trx_category"],
        "group_col": "cl_id",
        "truncation_pretrain": "sliding_window",
        "truncation_eval": "last_events",
        "amount_transform": "robust_scaler",
        "time_transform": "none",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "min_vocab_count": 5,
    },
    "corruption": {
        "event_level_masking": {"prob": 0.10},
        "event_type": {
            "selected_prob": 0.40,
            "mask_prob": 0.28,
            "transition_replace_prob": 0.00,
            "random_replace_prob": 0.10,
            "keep_predict_prob": 0.02,
            "use_transition_aware_replacement": False,
        },
        "categorical_features": {"mask_prob": 0.15, "random_replace_prob": 0.05},
        "time_noise": {
            "corruption_prob": 0.00,
            "min_std": 0.05,
            "max_std": 0.30,
            "sampling_level": "batch",
        },
        "numerical_noise": {
            "corruption_prob": 0.20,
            "min_std": 0.03,
            "max_std": 0.15,
            "sampling_level": "batch",
        },
    },
    "model": {
        "event_type_emb_dim": 64,
        "cat_emb_dim": 32,
        "num_projection_dim": 64,
        "time_projection_dim": 64,
        "hidden_dim": 256,
        "num_layers": 4,
        "num_heads": 8,
        "dim_feedforward": 1024,
        "dropout": 0.10,
        "activation": "gelu",
        "max_seq_len": 256,
    },
    "pooling": {"type": "gated_attention"},
    "loss": {
        "lambda_event_type": 1.0,
        "lambda_time": 1.0,
        "lambda_num": 0.5,
        "lambda_cat": 0.5,
        "lambda_exist": 0.1,
    },
    "training": {
        "batch_size": 128,
        "num_epochs_pretrain": 30,
        "num_epochs_finetune": 20,
        "lr": 3e-4,
        "lr_encoder": 3e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "gradient_clip_val": 1.0,
        "mixed_precision": USE_AMP,
        "early_stopping_patience": 5,
        "log_every_n_steps": 50,
    },
    "loss_calibration": {
        "enabled": True,
        "warmup_steps": 1000,
    },
}

CHECKPOINT_EVERY = 500

print("Config ready. Output dir:", OUTPUT_DIR)

In [ ]:
# Cell 3 — Data Loading & Preprocessing
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits

# Загрузка данных — adjust path to match the Kaggle dataset attachment
df = pd.read_csv(DATA_DIR / "rosbank" / "train.csv.gz")
df["TRDATETIME"] = pd.to_datetime(df["TRDATETIME"], format="%d%b%y:%H:%M:%S")
print(f"Raw data: {df.shape}")
print(df.head(3))

# Базовая валидация
assert config["data"]["event_type_col"] in df.columns, "event_type_col not found"
assert config["data"]["group_col"] in df.columns, "group_col not found"
n_entities = df[config["data"]["group_col"]].nunique()
print(f"Entities    : {n_entities}")
print(f"Events total: {len(df)}")

# Resume: переиспользовать препроцессор и сплиты если checkpoint существует
if PREP_PATH.exists() and SPLITS_PATH.exists():
    with open(PREP_PATH, "rb") as f:
        prep = pickle.load(f)
    with open(SPLITS_PATH) as f:
        splits = json.load(f)
    print("Loaded preprocessor and splits from checkpoint")
else:
    splits = make_entity_splits(
        df,
        entity_col=config["data"]["group_col"],
        target_col=config["data"]["target_col"],
        train_ratio=config["data"]["train_ratio"],
        val_ratio=config["data"]["val_ratio"],
        test_ratio=config["data"]["test_ratio"],
        seed=42,
    )
    prep = EventPreprocessor(config)
    prep.fit(df, splits["train"])
    with open(PREP_PATH, "wb") as f:
        pickle.dump(prep, f)
    with open(SPLITS_PATH, "w") as f:
        json.dump(splits, f, indent=2)
    print("Preprocessor fitted and saved")

print(f"Splits — train: {len(splits['train'])}, val: {len(splits['val'])}, test: {len(splits['test'])}")
print(f"Vocab sizes: event_type={len(prep.vocab[prep.event_type_col])}, "
      f"cat={[len(prep.vocab[c]) for c in prep.categorical_cols]}")

In [ ]:
# Cell 4 — Smoke Test: overfit на 1 batch
from src.corruption.pipeline import CorruptionPipeline
from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.models.dme_encoder import DMEEncoder
from src.training.losses import compute_pretraining_loss

# Маленькая конфигурация для smoke test
_smoke_cfg = {k: v for k, v in config.items()}
_smoke_cfg["model"] = dict(config["model"], hidden_dim=64, num_layers=2, num_heads=4, dim_feedforward=128)

vocab_info = {
    "event_type_vocab_size": len(prep.vocab[prep.event_type_col]),
    "cat_vocab_sizes": [len(prep.vocab[c]) for c in prep.categorical_cols],
    "num_num_features": len(prep.numerical_cols),
    "num_classes": 2,
}

_smoke_ds = EventSequenceDataset(df, splits["train"][:50], prep, _smoke_cfg, mode="pretrain")
_smoke_loader = DataLoader(_smoke_ds, batch_size=16, shuffle=False,
                           collate_fn=collate_fn, num_workers=0)
_smoke_batch = next(iter(_smoke_loader))
_smoke_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in _smoke_batch.items()}

_smoke_pipe = CorruptionPipeline(
    config=config["corruption"],
    vocab_sizes={
        "event_type": vocab_info["event_type_vocab_size"],
        "cat_features": vocab_info["cat_vocab_sizes"],
    },
)

_smoke_model = DMEEncoder(_smoke_cfg, vocab_info).to(device)
_smoke_opt = torch.optim.AdamW(_smoke_model.parameters(), lr=1e-3)

losses = []
_smoke_model.train()
for _ in range(20):
    corrupted, targets, masks = _smoke_pipe(_smoke_batch)
    masks["attention_mask"] = corrupted["attention_mask"]
    _smoke_opt.zero_grad()
    outputs = _smoke_model(corrupted, mode="pretrain")
    loss_dict = compute_pretraining_loss(outputs, targets, masks, _smoke_cfg)
    loss_dict["total"].backward()
    _smoke_opt.step()
    losses.append(loss_dict["total"].item())

print(f"Loss start : {losses[0]:.4f}")
print(f"Loss end   : {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss не уменьшился — что-то сломано!"
print("Smoke test passed — loss падает")

del _smoke_model, _smoke_opt, _smoke_batch, _smoke_ds, _smoke_loader

In [ ]:
# Cell 5 — Loss Calibration Warmup (1000 steps)
from src.training.pretrain import _log_calibration_recommendations

train_dataset = EventSequenceDataset(df, splits["train"], prep, config, mode="pretrain")
val_dataset   = EventSequenceDataset(df, splits["val"],   prep, config, mode="pretrain")

train_loader = DataLoader(train_dataset, batch_size=config["training"]["batch_size"],
                          shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=config["training"]["batch_size"],
                          shuffle=False, collate_fn=collate_fn, num_workers=0)

model = DMEEncoder(config, vocab_info).to(device)

pipe = CorruptionPipeline(
    config=config["corruption"],
    vocab_sizes={
        "event_type": vocab_info["event_type_vocab_size"],
        "cat_features": vocab_info["cat_vocab_sizes"],
    },
)

CAL_WARMUP_STEPS = config["loss_calibration"]["warmup_steps"]
_CAL_COMPONENTS = ["event_type", "time_delta", "numerical", "categorical", "existence"]
component_sums = {k: 0.0 for k in _CAL_COMPONENTS}

_cal_opt = torch.optim.AdamW(model.parameters(), lr=config["training"]["lr"],
                              weight_decay=config["training"]["weight_decay"])

model.train()
step = 0
LOG_EVERY_CAL = 100

print(f"Starting loss calibration warmup ({CAL_WARMUP_STEPS} steps)...")
for clean_batch in train_loader:
    if step >= CAL_WARMUP_STEPS:
        break
    clean_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                   for k, v in clean_batch.items()}
    corrupted, targets, masks = pipe(clean_batch)
    masks["attention_mask"] = corrupted["attention_mask"]
    _cal_opt.zero_grad()
    outputs = model(corrupted, mode="pretrain")
    loss_dict = compute_pretraining_loss(outputs, targets, masks, config)
    loss_dict["total"].backward()
    clip_grad_norm_(model.parameters(), config["training"]["gradient_clip_val"])
    _cal_opt.step()
    for k in _CAL_COMPONENTS:
        component_sums[k] += loss_dict[k].item()
    step += 1
    if step % LOG_EVERY_CAL == 0:
        print(f"  Step {step:4d}/{CAL_WARMUP_STEPS} | total={loss_dict['total'].item():.4f}")

_log_calibration_recommendations(component_sums, step, config)
del _cal_opt

In [ ]:
# Cell 6 — Full Pretraining
from src.training.optim import build_pretrain_optimizer
from src.training.pretrain import evaluate_pretrain

optimizer, scheduler = build_pretrain_optimizer(model, config)
scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

num_epochs    = config["training"]["num_epochs_pretrain"]
grad_clip     = config["training"]["gradient_clip_val"]
log_every     = config["training"]["log_every_n_steps"]
start_epoch   = 0
global_step   = 0
best_val_loss = float("inf")

# Resume logic
if STEP_CKPT_PATH.exists():
    ckpt = torch.load(STEP_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch   = ckpt["epoch"]
    global_step   = ckpt["global_step"]
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print(f"Resumed from step {global_step}, epoch {start_epoch}")
else:
    print("Starting fresh pretraining")

pretrain_history = []

for epoch in range(start_epoch, num_epochs):
    model.train()
    for clean_batch in train_loader:
        clean_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                       for k, v in clean_batch.items()}
        corrupted, targets, masks = pipe(clean_batch)
        masks["attention_mask"] = corrupted["attention_mask"]

        optimizer.zero_grad()

        if USE_AMP:
            with torch.autocast(device_type="cuda"):
                outputs  = model(corrupted, mode="pretrain")
                loss_dict = compute_pretraining_loss(outputs, targets, masks, config)
            scaler.scale(loss_dict["total"]).backward()
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs  = model(corrupted, mode="pretrain")
            loss_dict = compute_pretraining_loss(outputs, targets, masks, config)
            loss_dict["total"].backward()
            clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        global_step += 1

        if global_step % log_every == 0:
            l = loss_dict["total"].item()
            pretrain_history.append({"step": global_step, "epoch": epoch, "loss": l})
            print(f"Epoch {epoch:02d} | Step {global_step:5d} | Loss: {l:.4f}")

        # Checkpoint каждые 500 шагов — критично при Kaggle 9h timeout
        if global_step % CHECKPOINT_EVERY == 0:
            torch.save({
                "model_state_dict":     model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch":                epoch,
                "global_step":          global_step,
                "best_val_loss":        best_val_loss,
                "vocab_info":           vocab_info,
                "config":               config,
            }, STEP_CKPT_PATH)
            print(f"  [Step checkpoint saved at step {global_step}]")

    scheduler.step()

    val_metrics = evaluate_pretrain(model, val_loader, pipe, config, device)
    val_loss = val_metrics["loss_total"]
    print(f"Epoch {epoch:02d} — Val Loss: {val_loss:.4f} "
          f"| Type Acc: {val_metrics['event_type_accuracy']:.4f} "
          f"| Time MAE: {val_metrics['time_delta_mae']:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch":            epoch,
            "val_loss":         best_val_loss,
            "vocab_info":       vocab_info,
            "config":           config,
        }, FINAL_CKPT_PATH)
        print(f"  [Best model saved: val_loss={best_val_loss:.4f}]")

print(f"\nPretraining complete. Best val_loss: {best_val_loss:.4f}")
pd.DataFrame(pretrain_history).to_csv(OUTPUT_DIR / "pretrain_history.csv", index=False)

In [ ]:
# Cell 7 — Fine-tuning (linear probing + full fine-tuning)
from src.training.finetune import evaluate_finetune, finetune
from src.utils.logging import MetricsLogger

ft_train_ds = EventSequenceDataset(df, splits["train"], prep, config, mode="finetune")
ft_val_ds   = EventSequenceDataset(df, splits["val"],   prep, config, mode="eval")
ft_test_ds  = EventSequenceDataset(df, splits["test"],  prep, config, mode="eval")

ft_train_loader = DataLoader(ft_train_ds, batch_size=64, shuffle=True,
                             collate_fn=collate_fn, num_workers=0)
ft_val_loader   = DataLoader(ft_val_ds,   batch_size=64, shuffle=False,
                             collate_fn=collate_fn, num_workers=0)
ft_test_loader  = DataLoader(ft_test_ds,  batch_size=64, shuffle=False,
                             collate_fn=collate_fn, num_workers=0)

# --- Linear probing (замороженный энкодер) ---
print("=" * 60)
print("Linear probing (frozen encoder)")
print("=" * 60)
lp_model = DMEEncoder(config, vocab_info).to(device)
lp_logger = MetricsLogger(str(LOG_DIR), "kaggle_linear_probing")
lp_ckpt_path = finetune(
    model=lp_model,
    train_loader=ft_train_loader,
    val_loader=ft_val_loader,
    config=config,
    output_dir=str(CKPT_DIR),
    device=device,
    logger=lp_logger,
    pretrained_checkpoint=str(FINAL_CKPT_PATH),
    frozen_encoder=True,
    label_fraction=1.0,
    vocab_info=vocab_info,
)
lp_metrics = evaluate_finetune(lp_model, ft_test_loader, num_classes=2, device=device)
print("Linear probing test metrics:", lp_metrics)

# --- Full fine-tuning ---
print("=" * 60)
print("Full fine-tuning (all params)")
print("=" * 60)
ft_model = DMEEncoder(config, vocab_info).to(device)
ft_logger = MetricsLogger(str(LOG_DIR), "kaggle_finetune_full")
ft_ckpt_path = finetune(
    model=ft_model,
    train_loader=ft_train_loader,
    val_loader=ft_val_loader,
    config=config,
    output_dir=str(CKPT_DIR),
    device=device,
    logger=ft_logger,
    pretrained_checkpoint=str(FINAL_CKPT_PATH),
    frozen_encoder=False,
    label_fraction=1.0,
    vocab_info=vocab_info,
)
ft_metrics = evaluate_finetune(ft_model, ft_test_loader, num_classes=2, device=device)
print("Full fine-tuning test metrics:", ft_metrics)

In [ ]:
# Cell 8 — Low-label experiment
# Используем 3 fractions (для Kaggle: быстрый прогон)
LOW_LABEL_FRACTIONS = [0.05, 0.25, 1.00]
SEEDS = [42, 43, 44]

low_label_results = []

for frac in LOW_LABEL_FRACTIONS:
    frac_metrics_per_seed = []
    for seed in SEEDS:
        print(f"\n--- fraction={frac:.2f}, seed={seed} ---")
        ll_model  = DMEEncoder(config, vocab_info).to(device)
        ll_logger = MetricsLogger(str(LOG_DIR), f"kaggle_ll_f{int(frac*100):03d}_s{seed}")
        finetune(
            model=ll_model,
            train_loader=ft_train_loader,
            val_loader=ft_val_loader,
            config=dict(config, training=dict(config["training"],
                        num_epochs_finetune=10)),  # меньше эпох для скорости
            output_dir=str(CKPT_DIR),
            device=device,
            logger=ll_logger,
            pretrained_checkpoint=str(FINAL_CKPT_PATH),
            frozen_encoder=False,
            label_fraction=frac,
            vocab_info=vocab_info,
        )
        m = evaluate_finetune(ll_model, ft_test_loader, num_classes=2, device=device)
        m["fraction"] = frac
        m["seed"] = seed
        frac_metrics_per_seed.append(m)
        del ll_model

    # Среднее по seeds
    avg = {"fraction": frac, "seed": "mean"}
    for k in ("roc_auc", "pr_auc", "macro_f1", "balanced_accuracy"):
        avg[k] = float(np.mean([x[k] for x in frac_metrics_per_seed]))
    low_label_results.extend(frac_metrics_per_seed)
    low_label_results.append(avg)
    print(f"  fraction={frac:.2f} avg ROC-AUC={avg['roc_auc']:.4f}")

pd.DataFrame(low_label_results).to_csv(OUTPUT_DIR / "low_label_results.csv", index=False)
print("\nLow-label experiment complete")

In [ ]:
# Cell 9 — Results Table
all_results = []

# Supervised baseline (linear probing)
all_results.append({
    "experiment": "linear_probing",
    "fraction": 1.0,
    **lp_metrics,
})

# Full fine-tuning
all_results.append({
    "experiment": "full_finetune",
    "fraction": 1.0,
    **ft_metrics,
})

# Low-label means
for row in low_label_results:
    if row.get("seed") == "mean":
        all_results.append({
            "experiment": f"low_label_{row['fraction']:.0%}",
            "fraction": row["fraction"],
            "roc_auc": row["roc_auc"],
            "pr_auc": row["pr_auc"],
            "macro_f1": row["macro_f1"],
            "balanced_accuracy": row["balanced_accuracy"],
        })

results_df = pd.DataFrame(all_results)
metric_cols = [c for c in results_df.columns if c not in ("experiment", "fraction")]

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

try:
    display(results_df.style
            .highlight_max(subset=metric_cols, color="#d4edda")
            .format({c: "{:.4f}" for c in metric_cols if c in results_df.columns})
            .set_caption("DME-Encoder — Experiment Results"))
except NameError:
    print(results_df.to_string(index=False))

# Сохранение
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
(OUTPUT_DIR / "results.json").write_text(
    json.dumps(all_results, indent=2, ensure_ascii=False)
)
print(f"\nResults saved to {OUTPUT_DIR}")